## Imports

In [ ]:
import sys
import pickle
import json

import numpy as np
import pandas as pd

sys.path.append("../src")
from trust_library.core import TrustEvaluator
from trust_library.factsheet import Factsheet
from trust_library.utils import to_json_safe

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    classification_report, accuracy_score
)
from sklearn.datasets import fetch_california_housing, fetch_kddcup99

import joblib

# Deep Learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses, callbacks

## 1. Random Forest Classifier

**Modelo:** Clasificador Random Forest con 100 árboles

**Dataset:** Datos tabulares cargados desde `train.csv` y `test.csv`

**Objetivo:** Entrenar un modelo de clasificación binaria y guardarlo para su evaluación posterior con TrustEvaluator.

**Adaptación para TrustEvaluator:**
- El modelo se guarda con `pickle` para que pueda ser cargado directamente
- TrustEvaluator espera que el modelo tenga los métodos `predict()` y `predict_proba()`
- RandomForest de sklearn implementa ambos métodos nativamente

In [ ]:
# Cargar datos
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
X_train = train.drop(columns=["Target"])
y_train = train["Target"]

# Entrenar modelo
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Guardar modelo
with open("model.pkl", "wb") as f:
    pickle.dump(clf, f)

print("Modelo Random Forest guardado en: model.pkl")

## 2. Support Vector Machine (SVM)

**Modelo:** SVM con kernel RBF (Radial Basis Function)

**Dataset:** Datos tabulares cargados desde `train.csv` y `test.csv` (mismo dataset que Random Forest)

**Objetivo:** Entrenar un segundo modelo de clasificación para comparar rendimiento y características con Random Forest en TrustEvaluator.

**Adaptación para TrustEvaluator:**
- SVM también implementa nativamente `predict()` y `predict_proba()` (con `probability=True`)
- La salida de `predict_proba()` es necesaria para módulos como explainability y robustness en TrustEvaluator

In [ ]:
# Cargar datos
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
X_train = train.drop(columns=["Target"])
y_train = train["Target"]

# Entrenar modelo SVM
clf = SVC(
    kernel="rbf",
    probability=True,   # Necesario para predict_proba
    random_state=42
)
clf.fit(X_train, y_train)

# Guardar modelo
with open("model_SVM.pkl", "wb") as f:
    pickle.dump(clf, f)

print("Modelo SVM guardado en: model_SVM.pkl")

## 3. Decision Tree Classifier

**Modelo:** Árbol de Decisión con profundidad máxima de 5

**Dataset:** Datos tabulares cargados desde `train.csv` y `test.csv` (mismo dataset)

**Objetivo:** Entrenar un modelo interpretable de clasificación. Los árboles de decisión son especialmente útiles para evaluar explainability en TrustEvaluator.

**Adaptación para TrustEvaluator:**
- Decision Tree es altamente interpretable, ideal para módulos de explainability
- Implementa nativamente `predict()` y `predict_proba()`
- La profundidad limitada (max_depth=5) facilita la visualización y explicación del modelo

In [ ]:
# Cargar datos
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
X_train = train.drop(columns=["Target"])
y_train = train["Target"]

# Entrenar modelo
clf = DecisionTreeClassifier(random_state=42, max_depth=5)
clf.fit(X_train, y_train)

# Guardar modelo
with open("model_tree.pkl", "wb") as f:
    pickle.dump(clf, f)

print("Modelo Decision Tree guardado en: model_tree.pkl")

## 4. Red Neuronal Simple - PyTorch

**Modelo:** Red neuronal feed-forward para clasificación binaria de enfermedades cardíacas  
Arquitectura: Input(13) → 32 → 16 → 2 clases

**Dataset:** UCI Heart Disease Dataset  
- Descargado directamente desde URL
- 13 features médicas relacionadas con enfermedades cardíacas
- Target binario: 0 (sin enfermedad), 1 (con enfermedad)

**Objetivo:** Demostrar cómo entrenar y adaptar modelos de deep learning (PyTorch) para TrustEvaluator.

**Adaptación para TrustEvaluator:**
- El modelo se guarda como `estado de pesos` (.pth) junto con la configuración
- Creamos DataFrames con columnas normalizadas (pixel_0 a pixel_783 por convención)
- Se añaden columnas metadata: `Group` (fairness), `QI_1` y `QI_2` (privacy)
- TrustEvaluator cargará el modelo y lo envolverá con métodos `predict()` y `predict_proba()`

In [ ]:
# =========================================================================
# 1. DESCARGAR DATOS DEL DATASET HEART DISEASE
# =========================================================================

url_data = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach", 
           "exang", "oldpeak", "slope", "ca", "thal", "target"]
df = pd.read_csv(url_data, names=columns, na_values="?")
df = df.dropna()

# Binarizar el target (0 = sin enfermedad, 1 = con enfermedad)
df["target"] = (df["target"] > 0).astype(int)

X = df.drop("target", axis=1).values
y = df["target"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

train_df = pd.DataFrame(X_train, columns=columns[:-1])
train_df["target"] = y_train
test_df = pd.DataFrame(X_test, columns=columns[:-1])
test_df["target"] = y_test

train_df.to_csv("./train_heart.csv", index=False)
test_df.to_csv("./test_heart.csv", index=False)

# =========================================================================
# 2. DEFINIR RED NEURONAL
# =========================================================================

class HeartDiseaseNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 2)
        )
    
    def forward(self, x):
        return self.net(x)

# =========================================================================
# 3. ENTRENAR MODELO
# =========================================================================

pytorch_model = HeartDiseaseNet(input_dim=X_train.shape[1])
optimizer = torch.optim.Adam(pytorch_model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

# Escalar datos (vital para redes neuronales)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

# Entrenamiento
for epoch in range(100):
    optimizer.zero_grad()
    loss = criterion(pytorch_model(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()

# Guardar modelo
torch.save(pytorch_model.state_dict(), "./modelo_heart.pth")
print("Modelo PyTorch guardado en: ./modelo_heart.pth")

Modelo guardado como './modelo_corazon_real.pth'


## 5. Red Neuronal Convolucional Pesada - PyTorch

**Modelo:** CNN con 3 bloques convolucionales + BatchNorm + Dropout  
Parámetros: ~1.5M  
Objetivo de salida: 10 clases

**Dataset:** FashionMNIST  
- Imágenes 28x28 de ropa y accesorios
- 60,000 imágenes de entrenamiento, 10,000 de test
- Clases: T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot

**Objetivo:** Entrenar un modelo de deep learning más complejo con imágenes para evaluar robustness, fairness y otros pilares en modelos de visión por computadora.

**Adaptación para TrustEvaluator:**
- Las imágenes se **aplanan** a vectores de 784 dimensiones (28×28)
- Se normalizan a [0, 1]
- Se crean DataFrames con columnas `pixel_0` a `pixel_783`
- Se añade columna `Group` para fairness basada en categoría de prenda (prendas superiores vs inferiores)
- Se crean `QI_1` y `QI_2` simulando quasi-identificadores para evaluación de privacy
- El modelo necesita un wrapper para adaptar formato entrada/salida al esperado por TrustEvaluator

In [ ]:
# =========================================================================
# DEFINICIÓN DEL MODELO CNN (PyTorch)
# =========================================================================

class FashionMNISTCNN(nn.Module):
    """
    Red convolucional para FashionMNIST con ~1.5M parámetros.
    
    Arquitectura:
    - 3 bloques convolucionales con BatchNorm, ReLU y MaxPool
    - 3 capas fully connected con Dropout
    """
    def __init__(self, num_classes=10):
        super(FashionMNISTCNN, self).__init__()

        # Bloque convolucional 1
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25)
        )

        # Bloque convolucional 2
        self.conv2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25)
        )

        # Bloque convolucional 3
        self.conv3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25)
        )

        # Capas fully connected
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 3 * 3, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # Si entrada viene aplanada (desde TrustEvaluator), reshapear
        if x.dim() == 2:
            if x.shape[1] > 784:
                x = x[:, :784]
            x = x.view(-1, 1, 28, 28)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.fc(x)
        return x


# =========================================================================
# FUNCIONES DE ENTRENAMIENTO Y EVALUACIÓN
# =========================================================================

def train_model(model, train_loader, device, epochs=10, lr=0.001):
    """Entrena el modelo CNN."""
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0

        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

        scheduler.step()
        acc = 100. * correct / total
        avg_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f} - Acc: {acc:.2f}%")

    return model


def evaluate_model(model, test_loader, device):
    """Evalúa el modelo en test set."""
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

    accuracy = 100. * correct / total
    print(f"\nTest Accuracy: {accuracy:.2f}%")
    return accuracy


def prepare_dataframes(train_dataset, test_dataset, sample_size=5000):
    """
    Convierte datasets de PyTorch a DataFrames para TrustEvaluator.
    
    Columnas creadas:
    - pixel_0 a pixel_783: los 784 píxeles de cada imagen
    - target: clase predicha
    - Group: 0 para prendas superiores, 1 para inferiores/accesorios (fairness)
    - QI_1, QI_2: quasi-identificadores simulados (privacy)
    """
    train_images = train_dataset.data.numpy()
    train_labels = train_dataset.targets.numpy()
    test_images = test_dataset.data.numpy()
    test_labels = test_dataset.targets.numpy()

    if sample_size and sample_size < len(train_images):
        train_indices = np.random.choice(len(train_images), sample_size, replace=False)
        train_images = train_images[train_indices]
        train_labels = train_labels[train_indices]

    if sample_size and sample_size < len(test_images):
        test_indices = np.random.choice(len(test_images), sample_size, replace=False)
        test_images = test_images[test_indices]
        test_labels = test_labels[test_indices]

    # Aplanar imágenes
    train_flat = train_images.reshape(len(train_images), -1)
    test_flat = test_images.reshape(len(test_images), -1)

    # Normalizar a [0, 1]
    train_flat = train_flat / 255.0
    test_flat = test_flat / 255.0

    # Crear DataFrames
    feature_cols = [f"pixel_{i}" for i in range(784)]
    train_df = pd.DataFrame(train_flat, columns=feature_cols)
    test_df = pd.DataFrame(test_flat, columns=feature_cols)

    train_df["target"] = train_labels
    test_df["target"] = test_labels

    # Crear Group para fairness
    upper_clothes = [0, 2, 3, 4, 6]
    train_df["Group"] = train_df["target"].apply(lambda x: 0 if x in upper_clothes else 1)
    test_df["Group"] = test_df["target"].apply(lambda x: 0 if x in upper_clothes else 1)

    # Crear quasi-identificadores para privacy
    train_df["QI_1"] = (train_df["pixel_100"] * 10).astype(int)
    train_df["QI_2"] = (train_df["pixel_200"] * 10).astype(int)
    test_df["QI_1"] = (test_df["pixel_100"] * 10).astype(int)
    test_df["QI_2"] = (test_df["pixel_200"] * 10).astype(int)

    return train_df, test_df

In [ ]:
print("=" * 70)
print("ENTRENAMIENTO CNN - FASHIONMNIST (PyTorch)")
print("=" * 70)

# Configuración
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128
EPOCHS = 3
NUM_CLASSES = 10
SAMPLE_SIZE = 3000

print(f"\nDispositivo: {DEVICE}")

# Paso 1: Descargar FashionMNIST
print("\n1. Descargando FashionMNIST...")
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"   Train: {len(train_dataset)} imágenes")
print(f"   Test: {len(test_dataset)} imágenes")

# Paso 2: Crear y entrenar modelo
print("\n2. Entrenando modelo CNN...")
model = FashionMNISTCNN(num_classes=NUM_CLASSES)

try:
    model.load_state_dict(torch.load("./fashion_pytorch.pth"))
    print("   Modelo cargado desde ./fashion_pytorch.pth")
except FileNotFoundError:
    print("   Entrenando nuevo modelo...")
    model = train_model(model, train_loader, DEVICE, epochs=EPOCHS)
    torch.save(model.state_dict(), "./fashion_pytorch.pth")
    print("   Modelo guardado en ./fashion_pytorch.pth")

total_params = sum(p.numel() for p in model.parameters())
print(f"   Total parámetros: {total_params:,}")

model = model.to(DEVICE)
model.eval()

# Paso 3: Evaluar
print("\n3. Evaluando modelo...")
evaluate_model(model, test_loader, DEVICE)

# Paso 4: Preparar datos para TrustEvaluator
print(f"\n4. Preparando DataFrames para TrustEvaluator (muestra: {SAMPLE_SIZE})...")
train_df, test_df = prepare_dataframes(train_dataset, test_dataset, sample_size=SAMPLE_SIZE)
train_df.to_csv("./fashion_pytorch_train.csv", index=False)
test_df.to_csv("./fashion_pytorch_test.csv", index=False)

print(f"   Train: {train_df.shape}")
print(f"   Test: {test_df.shape}")

## 6. Red Neuronal Convolucional - TensorFlow/Keras

**Modelo:** Mismo modelo CNN que en PyTorch pero implementado en TensorFlow/Keras  
Parámetros: ~1.5M

**Dataset:** FashionMNIST (mismo que celda anterior)

**Objetivo:** Demostrar que TrustEvaluator puede evaluar modelos de diferentes frameworks (framework-agnostic).

**Adaptación para TrustEvaluator:**
- El modelo se guarda con `.save_weights()` en Keras
- La arquitectura es idéntica a la versión PyTorch para comparativas justas
- Los datos se preparan igual que en PyTorch: aplanados, normalizados, con Group y QI
- Keras también proporciona `predict()` y `predict_proba()` (en forma de softmax)

In [ ]:
# =========================================================================
# DEFINICIÓN DEL MODELO CNN (TensorFlow/Keras)
# =========================================================================

class FashionMNISTCNN_TF(tf.keras.Model):
    """
    Red convolucional para FashionMNIST usando Keras subclassing API.
    Misma arquitectura que la versión PyTorch.
    """
    def __init__(self, num_classes=10):
        super(FashionMNISTCNN_TF, self).__init__()

        # Bloque convolucional 1
        self.conv1_1 = layers.Conv2D(64, (3, 3), padding='same')
        self.bn1_1 = layers.BatchNormalization()
        self.act1_1 = layers.ReLU()
        self.conv1_2 = layers.Conv2D(64, (3, 3), padding='same')
        self.bn1_2 = layers.BatchNormalization()
        self.act1_2 = layers.ReLU()
        self.pool1 = layers.MaxPooling2D((2, 2))
        self.drop1 = layers.Dropout(0.25)

        # Bloque convolucional 2
        self.conv2_1 = layers.Conv2D(128, (3, 3), padding='same')
        self.bn2_1 = layers.BatchNormalization()
        self.act2_1 = layers.ReLU()
        self.conv2_2 = layers.Conv2D(128, (3, 3), padding='same')
        self.bn2_2 = layers.BatchNormalization()
        self.act2_2 = layers.ReLU()
        self.pool2 = layers.MaxPooling2D((2, 2))
        self.drop2 = layers.Dropout(0.25)

        # Bloque convolucional 3
        self.conv3_1 = layers.Conv2D(256, (3, 3), padding='same')
        self.bn3_1 = layers.BatchNormalization()
        self.act3_1 = layers.ReLU()
        self.conv3_2 = layers.Conv2D(256, (3, 3), padding='same')
        self.bn3_2 = layers.BatchNormalization()
        self.act3_2 = layers.ReLU()
        self.pool3 = layers.MaxPooling2D((2, 2))
        self.drop3 = layers.Dropout(0.25)

        # Capas fully connected
        self.flatten = layers.Flatten()
        self.fc1 = layers.Dense(512)
        self.bn_fc1 = layers.BatchNormalization()
        self.act_fc1 = layers.ReLU()
        self.drop_fc1 = layers.Dropout(0.5)
        self.fc2 = layers.Dense(256)
        self.bn_fc2 = layers.BatchNormalization()
        self.act_fc2 = layers.ReLU()
        self.drop_fc2 = layers.Dropout(0.5)
        self.classifier = layers.Dense(num_classes)

    def call(self, inputs, training=False):
        x = inputs
        
        # Si entrada viene aplanada (desde TrustEvaluator), reshapear
        if len(x.shape) == 2:
            if x.shape[1] > 784:
                x = x[:, :784]
            x = tf.reshape(x, [-1, 28, 28, 1])

        # Bloque 1
        x = self.conv1_1(x)
        x = self.bn1_1(x, training=training)
        x = self.act1_1(x)
        x = self.conv1_2(x)
        x = self.bn1_2(x, training=training)
        x = self.act1_2(x)
        x = self.pool1(x)
        x = self.drop1(x, training=training)

        # Bloque 2
        x = self.conv2_1(x)
        x = self.bn2_1(x, training=training)
        x = self.act2_1(x)
        x = self.conv2_2(x)
        x = self.bn2_2(x, training=training)
        x = self.act2_2(x)
        x = self.pool2(x)
        x = self.drop2(x, training=training)

        # Bloque 3
        x = self.conv3_1(x)
        x = self.bn3_1(x, training=training)
        x = self.act3_1(x)
        x = self.conv3_2(x)
        x = self.bn3_2(x, training=training)
        x = self.act3_2(x)
        x = self.pool3(x)
        x = self.drop3(x, training=training)

        # FC
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.bn_fc1(x, training=training)
        x = self.act_fc1(x)
        x = self.drop_fc1(x, training=training)
        x = self.fc2(x)
        x = self.bn_fc2(x, training=training)
        x = self.act_fc2(x)
        x = self.drop_fc2(x, training=training)

        logits = self.classifier(x)
        return tf.nn.softmax(logits)


def train_model_tf(model, x_train, y_train, epochs=10, batch_size=64, lr=0.001):
    """Entrena el modelo CNN usando Keras."""
    lr_schedule = optimizers.schedules.ExponentialDecay(
        initial_learning_rate=lr,
        decay_steps=(len(x_train) // batch_size) * 5,
        decay_rate=0.5,
        staircase=True
    )
    
    optimizer = optimizers.Adam(learning_rate=lr_schedule)
    loss_fn = losses.SparseCategoricalCrossentropy(from_logits=False)

    model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])
    model.fit(x_train, y_train, epochs=epochs, batch_size=batch_size, verbose=1)
    return model


def evaluate_model_tf(model, x_test, y_test):
    """Evalúa el modelo en test set."""
    loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
    print(f"\nTest Accuracy: {accuracy * 100:.2f}%")
    return accuracy


def prepare_dataframes_tf(sample_size=5000):
    """Prepara datos de FashionMNIST como DataFrames."""
    (train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.fashion_mnist.load_data()

    if sample_size and sample_size < len(train_images):
        train_indices = np.random.choice(len(train_images), sample_size, replace=False)
        train_images = train_images[train_indices]
        train_labels = train_labels[train_indices]

    if sample_size and sample_size < len(test_images):
        test_indices = np.random.choice(len(test_images), sample_size, replace=False)
        test_images = test_images[test_indices]
        test_labels = test_labels[test_indices]

    # Aplanar y normalizar
    train_flat = train_images.reshape(len(train_images), -1)
    test_flat = test_images.reshape(len(test_images), -1)
    train_flat = train_flat.astype("float32") / 255.0
    test_flat = test_flat.astype("float32") / 255.0

    # Crear DataFrames
    feature_cols = [f"pixel_{i}" for i in range(784)]
    train_df = pd.DataFrame(train_flat, columns=feature_cols)
    test_df = pd.DataFrame(test_flat, columns=feature_cols)

    train_df["target"] = train_labels
    test_df["target"] = test_labels

    # Group para fairness
    upper_clothes = [0, 2, 3, 4, 6]
    train_df["Group"] = train_df["target"].apply(lambda x: 0 if x in upper_clothes else 1)
    test_df["Group"] = test_df["target"].apply(lambda x: 0 if x in upper_clothes else 1)

    # Quasi-identificadores
    train_df["QI_1"] = (train_df["pixel_100"] * 10).astype(int)
    train_df["QI_2"] = (train_df["pixel_200"] * 10).astype(int)
    test_df["QI_1"] = (test_df["pixel_100"] * 10).astype(int)
    test_df["QI_2"] = (test_df["pixel_200"] * 10).astype(int)

    return train_df, test_df

In [ ]:
print("=" * 70)
print("ENTRENAMIENTO CNN - FASHIONMNIST (TensorFlow/Keras)")
print("=" * 70)

# Configuración
BATCH_SIZE = 128
EPOCHS = 3
NUM_CLASSES = 10
SAMPLE_SIZE = 3000

# Paso 1: Descargar FashionMNIST
print("\n1. Descargando FashionMNIST...")
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

# Normalizar y añadir dimensión de canal
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

print(f"   Train: {len(x_train)} imágenes")
print(f"   Test: {len(x_test)} imágenes")

# Paso 2: Crear y entrenar modelo
print("\n2. Entrenando modelo CNN...")
model = FashionMNISTCNN_TF(num_classes=NUM_CLASSES)
model = train_model_tf(model, x_train, y_train, epochs=EPOCHS)
model.save_weights("./fashion_tf.weights.h5")

total_params = model.count_params()
print(f"   Total parámetros: {total_params:,}")

# Paso 3: Evaluar
print("\n3. Evaluando modelo...")
evaluate_model_tf(model, x_test, y_test)

# Paso 4: Preparar datos para TrustEvaluator
print(f"\n4. Preparando DataFrames para TrustEvaluator (muestra: {SAMPLE_SIZE})...")
train_df, test_df = prepare_dataframes_tf(sample_size=SAMPLE_SIZE)
train_df.to_csv("./fashion_tf_train.csv", index=False)
test_df.to_csv("./fashion_tf_test.csv", index=False)

print(f"   Train: {train_df.shape}")
print(f"   Test: {test_df.shape}")

## 7. Modelos Sesgados y Justos - Regresión Logística

**Modelo:** Regresión Logística (dos instancias: una sesgada, una justa)

**Dataset:** Datos sintéticos de predicción de ingresos  
- 3,000 muestras
- 2 features numéricos: feature1, feature2
- 1 atributo sensible: sexo (0 = Hombre, 1 = Mujer)
- Target: ingresos binarios (0 = bajo, 1 = alto)

**Objetivo:** Demostrar cómo TrustEvaluator puede evaluar el sesgo en modelos.

**Adaptación para TrustEvaluator:**
- **Modelo Sesgado**: El éxito depende al 95% del sexo (variable sensible), ignorando el mérito
- **Modelo Justo**: El éxito depende únicamente de las features de mérito (feature1 + feature2)
- Ambos se guardan para ser evaluados con módulos de fairness
- Los datos incluyen la columna `sex` para análisis de disparidad de impacto

In [ ]:
print("=" * 70)
print("GENERACIÓN DE MODELOS SESGADOS Y JUSTOS")
print("=" * 70)

np.random.seed(42)
n_samples = 3000

# ===== ESCENARIO 1: MUY SESGADO =====
print("\n1. Creando modelo SESGADO...")

X_biased_f1 = np.random.normal(50, 10, n_samples)
X_biased_f2 = np.random.normal(50, 10, n_samples)
sex_biased = np.random.binomial(1, 0.5, n_samples)

# Sesgo: 95% éxito si Hombre, 5% si Mujer (ignorando features)
probs_biased = np.where(sex_biased == 0, 0.95, 0.05)
income_biased = np.random.binomial(1, probs_biased)

df_biased = pd.DataFrame({
    'feature1': X_biased_f1, 
    'feature2': X_biased_f2, 
    'sex': sex_biased, 
    'income': income_biased
})
train_biased, test_biased = train_test_split(df_biased, test_size=0.3, random_state=42)

train_biased.to_csv("./biased_train.csv", index=False)
test_biased.to_csv("./biased_test.csv", index=False)

X_train_b = train_biased.drop(columns=['income'])
y_train_b = train_biased['income']

model_biased = LogisticRegression()
model_biased.fit(X_train_b, y_train_b)

joblib.dump(model_biased, './model_biased.pkl')
print("   ✓ Modelo sesgado guardado")

# ===== ESCENARIO 2: JUSTO =====
print("\n2. Creando modelo JUSTO...")

X_fair_f1 = np.random.normal(50, 10, n_samples)
X_fair_f2 = np.random.normal(50, 10, n_samples)
sex_fair = np.random.binomial(1, 0.5, n_samples)

# Justicia: Éxito basado SOLO en el mérito (f1 + f2), sexo irrelevante
score = X_fair_f1 + X_fair_f2
probs_fair = 1 / (1 + np.exp(-(score - 100) / 5))
income_fair = np.random.binomial(1, probs_fair)

df_fair = pd.DataFrame({
    'feature1': X_fair_f1, 
    'feature2': X_fair_f2, 
    'sex': sex_fair, 
    'income': income_fair
})
train_fair, test_fair = train_test_split(df_fair, test_size=0.3, random_state=42)

train_fair.to_csv("./fair_train.csv", index=False)
test_fair.to_csv("./fair_test.csv", index=False)

X_train_f = train_fair.drop(columns=['income'])
y_train_f = train_fair['income']

model_fair = LogisticRegression()
model_fair.fit(X_train_f, y_train_f)

joblib.dump(model_fair, './model_fair.pkl')
print("   ✓ Modelo justo guardado")

## 8. Modelo de Regresión - Predicción de Precios de Viviendas

**Modelo:** Gradient Boosting Regressor con 200 árboles

**Dataset:** California Housing  
- 20,640 registros de viviendas en California
- 8 features: MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude
- Target: Valor medio de viviendas (en $100,000)

**Objetivo:** Demostrar que TrustEvaluator no solo actúa sobre clasificación, sino también sobre regresión.

**Adaptación para TrustEvaluator:**
- Los regressores requieren un wrapper especial (no tienen `predict_proba()`)
- Se crea `ScaledRegressorWrapper` que mantiene el scaler junto con el modelo
- Se añade `Group` basada en ingresos medios (bajo/alto) para fairness
- Se crean `QI_lat` y `QI_lon` como quasi-identificadores de ubicación (privacy)
- Las métricas de regresión se evalúan: MAE, MSE, RMSE, R²

In [ ]:
# =========================================================================
# FUNCIONES DE PREPARACIÓN Y ENTRENAMIENTO PARA REGRESIÓN
# =========================================================================

def load_and_prepare_data():
    """Carga California Housing dataset."""
    print("Descargando California Housing dataset...")
    housing = fetch_california_housing()
    df = pd.DataFrame(housing.data, columns=housing.feature_names)
    df['target'] = housing.target

    print(f"   {len(df)} registros, {len(housing.feature_names)} features")

    # Crear Group para fairness (basado en ingresos)
    median_income = df['MedInc'].median()
    df['Group'] = (df['MedInc'] > median_income).astype(int)

    # Crear quasi-identificadores para privacy
    df['QI_lat'] = pd.qcut(df['Latitude'], q=10, labels=False, duplicates='drop')
    df['QI_lon'] = pd.qcut(df['Longitude'], q=10, labels=False, duplicates='drop')

    # Separar train/test
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)


def get_feature_columns(df):
    """Obtiene columnas de features."""
    exclude = ['target', 'Group', 'QI_lat', 'QI_lon']
    return [c for c in df.columns if c not in exclude]


def train_model_regression(train_df, feature_cols):
    """Entrena modelo de regresión."""
    X_train = train_df[feature_cols]
    y_train = train_df['target']

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    print("\nEntrenando GradientBoosting Regressor...")
    model = GradientBoostingRegressor(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        min_samples_split=5, min_samples_leaf=3, subsample=0.8,
        random_state=42, verbose=0
    )
    model.fit(X_train_scaled, y_train)
    model.scaler_ = scaler
    model.feature_cols_ = feature_cols

    return model


def evaluate_model_regression(model, test_df, feature_cols):
    """Evalúa modelo de regresión."""
    X_test = test_df[feature_cols]
    y_test = test_df['target']

    X_test_scaled = model.scaler_.transform(X_test)
    y_pred = model.predict(X_test_scaled)

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"\nMétricas de Test:")
    print(f"   MSE:  {mse:.4f}")
    print(f"   RMSE: {rmse:.4f}")
    print(f"   MAE:  {mae:.4f}")
    print(f"   R²:   {r2:.4f}")

    # Feature importances
    print(f"\n   Importancia de features:")
    importances = model.feature_importances_
    for name, imp in sorted(zip(feature_cols, importances), key=lambda x: -x[1]):
        print(f"      {name:15s}: {imp:.4f}")

    return {'mse': mse, 'rmse': rmse, 'mae': mae, 'r2': r2}


class ScaledRegressorWrapper:
    """
    Wrapper que aplica scaler antes de predecir.
    Necesario porque TrustEvaluator pasa datos sin escalar.
    """
    def __init__(self, model, scaler, feature_cols):
        self.model = model
        self.scaler = scaler
        self.feature_cols = feature_cols
        self._is_regressor = True

    def predict(self, X):
        if isinstance(X, pd.DataFrame):
            X_features = X[self.feature_cols].values
        else:
            X_features = X[:, :len(self.feature_cols)]
        X_scaled = self.scaler.transform(X_features)
        return self.model.predict(X_scaled)

    def predict_proba(self, X):
        return None

Cargando dataset de Diabetes...
Archivos 'train_regression.csv' y 'test_regression.csv' guardados exitosamente.
Entrenando el modelo de Regresión Lineal...
Modelo entrenado. MSE: 2900.19 | R2 Score: 0.45



In [ ]:
print("=" * 70)
print("EVALUACIÓN DE TRUST - REGRESIÓN (CALIFORNIA HOUSING)")
print("=" * 70)

# Paso 1: Cargar y preparar datos
print("\n1. Cargando dataset California Housing...")
train_df, test_df = load_and_prepare_data()
train_df.to_csv("./california_housing_train.csv", index=False)
test_df.to_csv("./california_housing_test.csv", index=False)
feature_cols = get_feature_columns(train_df)

# Paso 2: Entrenar modelo
print("\n2. Entrenando modelo...")
model = train_model_regression(train_df, feature_cols)

# Paso 3: Evaluar modelo
print("\n3. Evaluando modelo...")
metrics = evaluate_model_regression(model, test_df, feature_cols)

# Guardar modelo
joblib.dump(model, './model_regression.pkl')
print("\n   ✓ Modelo guardado en: ./model_regression.pkl")

## 9. Clasificación Multiclase - Detección de Ataques de Red

**Modelo:** Random Forest con 100 árboles, balanceo de clases

**Dataset:** KDD Cup 1999 (versión 10%) - Ciberseguridad  
- 30,000 registros de tráfico de red
- 41 features (duración, tipo protocolo, bytes, flags, etc.)
- 10 clases: normal + 9 tipos de ataque (neptune, smurf, satan, etc.)

**Objetivo:** Demostrar cómo TrustEvaluator handle problemas multiclase complejos en casos reales (ciberseguridad).

**Adaptación para TrustEvaluator:**
- Datos categóricos se codifican con LabelEncoder
- Se aplica StandardScaler para normalización
- Se crea `Group` para fairness: 0 = tráfico normal, 1 = ataque
- Se crean `QI_src_bytes` y `QI_dst_bytes` como quasi-identificadores (privacy)
- El wrapper `ScaledModelWrapper` adapta formato entrada/salida
- TrustEvaluator puede evaluar métrica como accuracy overall y por clase

In [ ]:
# =========================================================================
# MAPEO Y CONSTANTES PARA KDD CUP 99
# =========================================================================

ATTACK_CATEGORIES = {
    b'normal.': 0, b'neptune.': 1, b'smurf.': 2, b'back.': 3, b'teardrop.': 4,
    b'satan.': 5, b'ipsweep.': 6, b'portsweep.': 7, b'nmap.': 8, b'warezclient.': 9,
}

CLASS_NAMES = [
    'normal', 'neptune_dos', 'smurf_dos', 'back_dos', 'teardrop_dos',
    'satan_probe', 'ipsweep_probe', 'portsweep_probe', 'nmap_scan', 'warezclient_r2l'
]

FEATURE_NAMES = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate'
]


# =========================================================================
# FUNCIONES DE PREPARACIÓN Y ENTRENAMIENTO PARA MULTICLASE
# =========================================================================

def load_and_prepare_data_multiclass(sample_size=30000):
    """Carga y prepara KDD Cup 99 para clasificación multiclase."""
    print("Descargando KDD Cup 99 dataset...")
    kdd = fetch_kddcup99(subset='SA', percent10=True, random_state=42)

    df = pd.DataFrame(kdd.data, columns=FEATURE_NAMES)
    df['attack_type'] = kdd.target

    print(f"   Dataset original: {len(df)} registros")

    # Filtrar solo clases deseadas
    df = df[df['attack_type'].isin(ATTACK_CATEGORIES.keys())]
    print(f"   Después de filtrar: {len(df)} registros")

    # Mapear a categorías numéricas
    df['target'] = df['attack_type'].map(ATTACK_CATEGORIES)
    df = df.drop(columns=['attack_type'])

    # Submuestrear
    if sample_size and len(df) > sample_size:
        df = df.sample(n=sample_size, random_state=42)
        print(f"   Después de submuestrear: {len(df)} registros")

    # Filtrar clases con pocos ejemplos
    class_counts = df['target'].value_counts()
    valid_classes = class_counts[class_counts >= 10].index.tolist()
    df = df[df['target'].isin(valid_classes)]

    # Remapear clases a consecutivos
    old_to_new = {old: new for new, old in enumerate(sorted(valid_classes))}
    df['target'] = df['target'].map(old_to_new)

    print(f"   Después de filtrar clases raras: {len(df)} registros")

    # Codificar variables categóricas
    categorical_cols = ['protocol_type', 'service', 'flag']
    for col in categorical_cols:
        df[col] = df[col].apply(lambda x: x.decode() if isinstance(x, bytes) else x)
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

    # Convertir a float
    for col in FEATURE_NAMES:
        df[col] = pd.to_numeric(df[col])

    # Crear Group (normal vs ataque)
    df['Group'] = (df['target'] != 0).astype(int)

    # Crear quasi-identificadores
    df['QI_src_bytes'] = pd.qcut(df['src_bytes'], q=10, labels=False, duplicates='drop')
    df['QI_dst_bytes'] = pd.qcut(df['dst_bytes'], q=10, labels=False, duplicates='drop')

    # Train/test split
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['target'])

    print(f"\n   Train: {len(train_df)} registros")
    print(f"   Test: {len(test_df)} registros")
    print(f"   Clases: {df['target'].nunique()}")

    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)


def get_feature_columns_multiclass(df):
    """Obtiene columnas de features exclusivamente."""
    exclude = ['target', 'Group', 'QI_src_bytes', 'QI_dst_bytes']
    return [c for c in df.columns if c not in exclude]


def train_model_multiclass(train_df, feature_cols):
    """Entrena modelo RandomForest para multiclase."""
    X_train = train_df[feature_cols]
    y_train = train_df['target']

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    print("\nEntrenando RandomForest para clasificación multiclase...")
    model = RandomForestClassifier(
        n_estimators=100, max_depth=20, min_samples_split=5, min_samples_leaf=2,
        n_jobs=-1, random_state=42, class_weight='balanced'
    )

    model.fit(X_train_scaled, y_train)
    model.scaler_ = scaler
    model.feature_cols_ = feature_cols

    return model


def evaluate_model_multiclass(model, test_df, feature_cols):
    """Evalúa modelo multiclase."""
    X_test = test_df[feature_cols]
    y_test = test_df['target']

    X_test_scaled = model.scaler_.transform(X_test)
    y_pred = model.predict(X_test_scaled)

    accuracy = accuracy_score(y_test, y_pred)
    print(f"\nTest Accuracy: {accuracy:.4f}")
    print("\nClassification Report:")

    present_classes = sorted(y_test.unique())
    present_names = [CLASS_NAMES[i] if i < len(CLASS_NAMES) else f"Class_{i}" 
                     for i in present_classes]

    print(classification_report(y_test, y_pred, labels=present_classes,
                                target_names=present_names, zero_division=0))

    return accuracy


class ScaledModelWrapper:
    """Wrapper que aplica scaler antes de predecir."""
    def __init__(self, model, scaler, feature_cols):
        self.model = model
        self.scaler = scaler
        self.feature_cols = feature_cols
        self.classes_ = model.classes_

    def predict(self, X):
        if isinstance(X, pd.DataFrame):
            X_features = X[self.feature_cols].values
        else:
            X_features = X[:, :len(self.feature_cols)]
        X_scaled = self.scaler.transform(X_features)
        return self.model.predict(X_scaled)

    def predict_proba(self, X):
        if isinstance(X, pd.DataFrame):
            X_features = X[self.feature_cols].values
        else:
            X_features = X[:, :len(self.feature_cols)]
        X_scaled = self.scaler.transform(X_features)
        return self.model.predict_proba(X_scaled)

Cargando dataset Wine...
Archivos 'train_multiclass.csv' y 'test_multiclass.csv' guardados exitosamente.
Entrenando el modelo RandomForestClassifier...
Modelo entrenado. Accuracy: 1.00
Reporte de clasificación:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       1.00      1.00      1.00        14
           2       1.00      1.00      1.00        10

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36



In [ ]:
print("=" * 70)
print("EVALUACIÓN DE TRUST - CLASIFICACIÓN MULTICLASE (KDD CUP 99)")
print("=" * 70)

# Paso 1: Cargar y preparar datos
print("\n1. Cargando dataset KDD Cup 99...")
train_df, test_df = load_and_prepare_data_multiclass(sample_size=30000)
feature_cols = get_feature_columns_multiclass(train_df)

train_df.to_csv("./multiclass_train.csv", index=False)
test_df.to_csv("./multiclass_test.csv", index=False)

# Paso 2: Entrenar modelo
print("\n2. Entrenando modelo...")
model = train_model_multiclass(train_df, feature_cols)

# Paso 3: Evaluar modelo
print("\n3. Evaluando modelo...")
accuracy = evaluate_model_multiclass(model, test_df, feature_cols)

# Guardar modelo
joblib.dump(model, './model_multiclass.pkl')
print("\n   ✓ Modelo guardado en: ./model_multiclass.pkl")